In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Create a DataFrame from your data
data = {
    'ADC': [65007, 63007, 54301, 49484, 40489, 32727, 24950, 15987, 10866, 3952, 2496, 512],
    'Lux': [0.1, 1, 10, 20, 50, 100, 200, 501, 1000, 5000, 10000, 100000]
}
df = pd.DataFrame(data)

# Create the plot
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))
sns.scatterplot(y='Lux', x='ADC', data=df)

# Set the labels
plt.ylabel('Beleuchtungsstärke in Lux')
plt.xlabel('ADC Wert')


plt.show()

: 

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# Messdaten Fotowiderstand
data = {
    'ADC': [65007, 63007, 54301, 49484, 40489, 32727, 24950, 15987, 10866, 3952, 2496, 512],
    'Lux': [0.1, 1, 10, 20, 50, 100, 200, 501, 1000, 5000, 10000, 100000]
}
df = pd.DataFrame(data)
lux = np.array(data['Lux'], dtype=float)
adc = np.array(data['ADC'], dtype=float)

# --- Modell: Potenzgesetz ----------------------------------
# ADC = a * Lux^(-gamma) + offset
def power_law(x, a, gamma, offset):
    return a * np.power(x, -gamma) + offset

p0 = [20000, 0.5, 0]
popt, pcov = curve_fit(power_law, lux, adc, p0=p0, maxfev=10000)
a, gamma, offset = popt
perr = np.sqrt(np.diag(pcov))

adc_pred = power_law(lux, *popt)
ss_res = np.sum((adc - adc_pred)**2)
ss_tot = np.sum((adc - np.mean(adc))**2)
r2 = 1 - ss_res / ss_tot

print("=" * 60)
print("Modell: Potenzgesetz")
print("  ADC = a · Lux^(-γ) + offset")
print(f"  a      = {a:.1f} ± {perr[0]:.1f}")
print(f"  γ      = {gamma:.4f} ± {perr[1]:.4f}")
print(f"  offset = {offset:.1f} ± {perr[2]:.1f}")
print(f"  R² = {r2:.5f}")
print()

# --- Plot ---------------------------------------------------
plt.figure(figsize=(10, 6))
lux_fine = np.logspace(np.log10(0.1), np.log10(100000), 500)

plt.scatter(lux, adc, c='black', s=60, zorder=5, label='Messdaten')
plt.plot(lux_fine, power_law(lux_fine, *popt), 'b-', lw=2,
        label=f'Potenzgesetz Fit (R²={r2:.4f})')
plt.xscale('log')
plt.xlabel('Beleuchtungsstärke [Lux]')
plt.ylabel('ADC-Wert')
plt.title('Fotowiderstand: Potenzgesetz Modell')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('photoresistor_powerlaw_fit.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot gespeichert als 'photoresistor_powerlaw_fit.png'")